In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [46]:
data_path = Path('../../data/train_clean.csv')
# keep_default_na=False desactiva la interpretación automática de nulos
# na_values=[] lista vacía — ningún valor adicional se interpreta como nulo
df = pd.read_csv(data_path, keep_default_na=False, na_values=[''])

print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Nulos totales: {df.isnull().sum().sum()}')

Filas: 1460
Columnas: 81
Nulos totales: 0


# Variables Seleccionadas para el Modelo
 
## Variables Numéricas Seleccionadas
Variables que se usan directamente sin transformación adicional.

| Variable | Descripción | Correlación con SalePrice |
|---|---|---|
| OverallQual | Calidad general de materiales y acabados (1-10) | 0.79 |
| GrLivArea | Área habitable sobre el suelo en pies cuadrados | 0.71 |
| GarageArea | Área del garaje en pies cuadrados | 0.62 |
| TotalBsmtSF | Área total del sótano en pies cuadrados | 0.61 |
| FullBath | Número de baños completos sobre el suelo | 0.56 |
| YearBuilt | Año de construcción original | 0.52 |
| YearRemodAdd | Año de última remodelación | 0.51 |
| MasVnrArea | Área de revestimiento de mampostería en pies cuadrados | 0.48 |
| Fireplaces | Número de chimeneas | 0.47 |
| BsmtFinSF1 | Área terminada del sótano tipo 1 en pies cuadrados | 0.39 |
| LotFrontage | Pies lineales de calle conectados a la propiedad | 0.35 |
| WoodDeckSF | Área de terraza de madera en pies cuadrados | 0.32 |
| 2ndFlrSF | Área del segundo piso en pies cuadrados | 0.32 |
| OpenPorchSF | Área de porche abierto en pies cuadrados | 0.32 |
| HalfBath | Número de medios baños sobre el suelo | 0.28 |

**Variables eliminadas por multicolinealidad:**
- `GarageCars` → redundante con `GarageArea`
- `TotRmsAbvGrd` → redundante con `GrLivArea`
- `GarageYrBlt` → redundante con `YearBuilt`
- `1stFlrSF` → redundante con `TotalBsmtSF`

---

## Variables Categóricas — Ordinal Encoding
Variables con jerarquía clara entre categorías. Se asigna un valor 
numérico respetando el orden de menor a mayor calidad o condición.

| Variable | Descripción | Orden |
|---|---|---|
| ExterQual | Calidad de materiales del exterior | Po < Fa < TA < Gd < Ex |
| KitchenQual | Calidad de la cocina | Po < Fa < TA < Gd < Ex |
| BsmtQual | Calidad del sótano (altura) | None < Po < Fa < TA < Gd < Ex |
| HeatingQC | Calidad del sistema de calefacción | Po < Fa < TA < Gd < Ex |
| BsmtExposure | Exposición del sótano al exterior | None < No < Mn < Av < Gd |
| BsmtFinType1 | Calidad del área terminada del sótano | None < Unf < LwQ < Rec < BLQ < ALQ < GLQ |
| GarageFinish | Acabado interior del garaje | None < Unf < RFn < Fin |
| PavedDrive | Tipo de entrada vehicular | N < P < Y |
| LotShape | Forma general del lote | IR3 < IR2 < IR1 < Reg |

---

## Variables Categóricas — One-Hot Encoding
Variables nominales sin orden jerárquico entre categorías. Se crean 
columnas binarias independientes por cada categoría para evitar 
introducir relaciones matemáticas artificiales.

| Variable | Descripción | # Categorías |
|---|---|---|
| Foundation | Tipo de cimentación | 6 |
| GarageType | Ubicación del garaje | 6 |
| MSZoning | Clasificación de zonificación general | 5 |
| SaleCondition | Condición de la venta | 6 |

---

## Variables Categóricas — Binary Encoding
Variables con exactamente 2 categorías. Se codifican como 0 y 1 
sin necesidad de One-Hot Encoding.

| Variable | Descripción | Codificación |
|---|---|---|
| CentralAir | Aire acondicionado central | N=0, Y=1 |

---

## Variables Categóricas — Ordinal Encoding Agrupado
Variables con alta cardinalidad que requieren agrupación previa 
por precio mediano antes de aplicar Ordinal Encoding.

| Variable | Descripción | # Categorías originales |
|---|---|---|
| Neighborhood | Ubicación dentro de Ames | 25 → 5 segmentos |

**Segmentos de Neighborhood por precio mediano:**
| Segmento | Valor | Vecindarios | Precio Mediano (USD) |
|---|---|---|---|
| Premium | 5 | NridgHt, NoRidge, StoneBr | 278,000 - 315,000 |
| Alto | 4 | Timber, Somerst, Veenker, Crawfor, ClearCr | 200,624 - 228,475 |
| Medio | 3 | CollgCr, Blmngtn, NWAmes, Gilbert, SawyerW | 179,900 - 197,200 |
| Bajo | 2 | Mitchel, NPkVill, NAmes, SWISU, Blueste, Sawyer | 135,000 - 153,500 |
| Muy Bajo | 1 | BrkSide, Edwards, OldTown, BrDale, IDOTRR, MeadowV | 88,000 - 124,300 |

---

## Target
| Variable | Descripción |
|---|---|
| SalePrice_log | Logaritmo natural del precio de venta — target del modelo |

# 1. Ordinal Encoding

In [ ]:

variables_calidad = ['ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC']

orden_variables_calidad = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
orden_bsmt_exposure = ['None', 'No', 'Mn', 'Av', 'Gd']
orden_bsmt_fin_type_1 = ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
orden_garage_finish = ['None', 'Unf', 'RFn', 'Fin']
orden_paved_drive = ['None','N','P','Y']
orden_lot_shape = ['IR3','IR2','IR1','Reg']

# Se debe enviar un orden para cada columna, para evitar repetir se multiplican las categorias por la cantidad de variables
encoder_qual = OrdinalEncoder(
    categories=[orden_variables_calidad] * len(variables_calidad)
)
# Encoder para las demas variables que no son de calidad
encoder_bsmt_exposure = OrdinalEncoder(categories=[orden_bsmt_exposure])
encoder_bsmt_fin_type_1 = OrdinalEncoder(categories=[orden_bsmt_fin_type_1])
encoder_garage_finish = OrdinalEncoder(categories=[orden_garage_finish])
encoder_paved_drive = OrdinalEncoder(categories=[orden_paved_drive])
encoder_lot_shape = OrdinalEncoder(categories=[orden_lot_shape])

## Fit aprende las categorias y transform aplica la transformacion usando lo que aprendio
## Espera una estructura de 2D no de 1D
df[variables_calidad] = encoder_qual.fit_transform(df[variables_calidad])
df['BsmtExposure'] = encoder_bsmt_exposure.fit_transform(df[['BsmtExposure']])
df['BsmtFinType1'] = encoder_bsmt_fin_type_1.fit_transform(df[['BsmtFinType1']])
df['GarageFinish'] = encoder_garage_finish.fit_transform(df[['GarageFinish']])
df['PavedDrive'] = encoder_paved_drive.fit_transform(df[['PavedDrive']])
df['LotShape'] = encoder_lot_shape.fit_transform(df[['LotShape']])

# Verificamos que todas las variables fueron codificadas correctamente
for var in variables_calidad:
    print(f'\n{var}:')
    print(df[var].value_counts().sort_index())

for var in ['BsmtExposure', 'BsmtFinType1', 'GarageFinish', 'PavedDrive', 'LotShape']:
    print(f'\n{var}:')
    print(df[var].value_counts().sort_index())

['Gd' 'TA' 'Ex' 'Fa']

ExterQual:
ExterQual
2.0     14
3.0    906
4.0    488
5.0     52
Name: count, dtype: int64

KitchenQual:
KitchenQual
2.0     39
3.0    735
4.0    586
5.0    100
Name: count, dtype: int64

BsmtQual:
BsmtQual
0.0     37
2.0     35
3.0    649
4.0    618
5.0    121
Name: count, dtype: int64

HeatingQC:
HeatingQC
1.0      1
2.0     49
3.0    428
4.0    241
5.0    741
Name: count, dtype: int64

BsmtExposure:
BsmtExposure
0.0     38
1.0    953
2.0    114
3.0    221
4.0    134
Name: count, dtype: int64

BsmtFinType1:
BsmtFinType1
0.0     37
1.0    430
2.0     74
3.0    133
4.0    148
5.0    220
6.0    418
Name: count, dtype: int64

GarageFinish:
GarageFinish
0.0     81
1.0    605
2.0    422
3.0    352
Name: count, dtype: int64

PavedDrive:
PavedDrive
1.0      90
2.0      30
3.0    1340
Name: count, dtype: int64

LotShape:
LotShape
0.0     10
1.0     41
2.0    484
3.0    925
Name: count, dtype: int64


# 2. One Hot Encoding

In [50]:
variables_onehot = ['Foundation', 'GarageType', 'MSZoning', 'SaleCondition']

# drop='first' elimina la primera categoría de cada variable
# sparse_output=False retorna un array denso en lugar de matriz dispersa
# handle_unknown='ignore' ignora categorías no vistas durante el fit
encoder_onehot = OneHotEncoder(
    drop='first',
    sparse_output=False,
    handle_unknown='ignore'
)

# Aplicamos el encoding
encoded_array = encoder_onehot.fit_transform(df[variables_onehot])

# Obtenemos los nombres de las columnas generadas
columnas_nuevas = encoder_onehot.get_feature_names_out(variables_onehot)

# Creamos un DataFrame con las columnas nuevas
df_encoded = pd.DataFrame(encoded_array, columns=columnas_nuevas, index=df.index)

# Unimos al DataFrame original y eliminamos las columnas originales
df = pd.concat([df.drop(columns=variables_onehot), df_encoded], axis=1)

# Verificamos cuántas columnas tenemos ahora
print(f'Columnas totales: {df.shape[1]}')
print(f'Columnas nuevas generadas: {columnas_nuevas}')

Columnas totales: 97
Columnas nuevas generadas: ['Foundation_CBlock' 'Foundation_PConc' 'Foundation_Slab'
 'Foundation_Stone' 'Foundation_Wood' 'GarageType_Attchd'
 'GarageType_Basment' 'GarageType_BuiltIn' 'GarageType_CarPort'
 'GarageType_Detchd' 'GarageType_None' 'MSZoning_FV' 'MSZoning_RH'
 'MSZoning_RL' 'MSZoning_RM' 'SaleCondition_AdjLand'
 'SaleCondition_Alloca' 'SaleCondition_Family' 'SaleCondition_Normal'
 'SaleCondition_Partial']


In [51]:
for var in variables_onehot:
    print(f'{var} en df: {var in df.columns}')

Foundation en df: False
GarageType en df: False
MSZoning en df: False
SaleCondition en df: False


# 3. Binary Encoding

In [52]:
# map() aplica un diccionario de mapeo a cada valor de la columna
df['CentralAir'] = df['CentralAir'].map({'N': 0, 'Y': 1})

print(df['CentralAir'].value_counts())

CentralAir
1    1365
0      95
Name: count, dtype: int64


# 4. Ordinal Encoding Agrupado

In [ ]:
neighborhood_segmentos = {
    'NridgHt': 5, 'NoRidge': 5, 'StoneBr': 5,
    'Timber': 4, 'Somerst': 4, 'Veenker': 4,
    'Crawfor': 4, 'ClearCr': 4,
    'CollgCr': 3, 'Blmngtn': 3, 'NWAmes': 3,
    'Gilbert': 3, 'SawyerW': 3,
    'Mitchel': 2, 'NPkVill': 2, 'NAmes': 2,
    'SWISU': 2, 'Blueste': 2, 'Sawyer': 2,
    'BrkSide': 1, 'Edwards': 1, 'OldTown': 1,
    'BrDale': 1, 'IDOTRR': 1, 'MeadowV': 1
}

df['Neighborhood'] = df['Neighborhood'].map(neighborhood_segmentos)
print(df['Neighborhood'].value_counts())

In [54]:
print(df['Neighborhood'].value_counts())

Neighborhood
2    384
3    378
1    341
4    214
5    143
Name: count, dtype: int64
